<a href="https://colab.research.google.com/github/keondreblankson/project2/blob/main/project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS 362: Three-Dataset Cleaning and Analysis Project

This project uses three historical datasets: U.S. personal spending, world telephone counts, and Virginia death rates. I started with each dataset in a wide format, read the information from CSV files, and used pandas to clean and reshape the data. I then analyzed each dataset and explained what I found.

I selected these datasets under the assignment’s option to use our own datasets. The analysis questions below are questions I chose for these datasets.

All original measurements were preserved. Only the first column heading was renamed to describe its contents.

The notebook includes copies of the downloaded data and creates the CSV files before reading them into pandas. This allows the notebook to run without uploading separate files.

In [ ]:
from pathlib import Path
from io import StringIO
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 60)

data_dir = Path.cwd()

print("pandas version:", pd.__version__)

pandas version: 2.2.3


In [ ]:
source_snapshots = {
    "us_spending_wide.csv": """Category,1940,1945,1950,1955,1960
Food and Tobacco,22.2,44.5,59.6,73.2,86.8
Household Operation,10.5,15.5,29,36.5,46.2
Medical and Health,3.53,5.76,9.71,14,21.1
Personal Care,1.04,1.98,2.45,3.4,5.4
Private Education,0.341,0.974,1.8,2.6,3.64
""",

    "world_telephones_wide.csv": """Year,N.Amer,Europe,Asia,S.Amer,Oceania,Africa,Mid.Amer
1951,45939,21574,2876,1815,1646,89,555
1956,60423,29990,4708,2568,2366,1411,733
1957,64721,32510,5230,2695,2526,1546,773
1958,68484,35218,6662,2845,2691,1663,836
1959,71799,37598,6856,3000,2868,1769,911
1960,76036,40341,8220,3145,3054,1905,1008
1961,79831,43173,9053,3338,3224,2005,1076
""",

    "virginia_death_rates_wide.csv": """Age_Group,Rural Male,Rural Female,Urban Male,Urban Female
50-54,11.7,8.7,15.4,8.4
55-59,18.1,11.7,24.3,13.6
60-64,26.9,20.3,37,19.3
65-69,41,30.9,54.6,35.1
70-74,66,54.3,71.1,50
"""
}

for filename, csv_text in source_snapshots.items():
    path = data_dir / filename

    if not path.exists():
        path.write_text(csv_text, encoding="utf-8")

    actual = pd.read_csv(path)
    expected = pd.read_csv(StringIO(csv_text))

    pd.testing.assert_frame_equal(actual, expected)

    print(
        f"{filename}: {actual.shape[0]} rows, "
        f"{actual.shape[1]} columns; source copy verified"
    )


def check_wide(frame, label):
    print(label)
    print("Shape:", frame.shape)
    print("Missing values:", int(frame.isna().sum().sum()))
    print("Duplicate rows:", int(frame.duplicated().sum()))
    print(frame.to_string(index=False))

    assert not frame.isna().any().any()
    assert not frame.duplicated().any()

us_spending_wide.csv: 5 rows, 6 columns; source copy verified
world_telephones_wide.csv: 7 rows, 8 columns; source copy verified
virginia_death_rates_wide.csv: 5 rows, 5 columns; source copy verified


## Dataset 1: U.S. Personal Spending, 1940–1960

This dataset includes five spending categories for 1940, 1945, 1950, 1955, and 1960. The amounts are measured in billions of dollars.

Question: Which category had the largest percentage increase from 1940 to 1960, and which category had the highest spending in 1960?

Online CSV:
https://vincentarelbundock.github.io/Rdatasets/csv/datasets/USPersonalExpenditure.csv

Official R documentation:
https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/USPersonalExpenditure.html

The documentation identifies The World Almanac and Book of Facts, 1962, page 756, as the original source.

In [ ]:
spending_wide = pd.read_csv(data_dir / "us_spending_wide.csv")

check_wide(
    spending_wide,
    "Original wide spending dataset"
)

Original wide spending dataset
Shape: (5, 6)
Missing values: 0
Duplicate rows: 0
           Category   1940   1945  1950  1955  1960
   Food and Tobacco 22.200 44.500 59.60  73.2 86.80
Household Operation 10.500 15.500 29.00  36.5 46.20
 Medical and Health  3.530  5.760  9.71  14.0 21.10
      Personal Care  1.040  1.980  2.45   3.4  5.40
  Private Education  0.341  0.974  1.80   2.6  3.64


### Cleanup and Transformation

The original dataset has separate columns for each year. I used melt() to put those years into one Year column and created a separate Spending_Billions column for the amounts.

I also cleaned the category labels, converted the years to integers, converted the spending amounts to numeric values, and sorted the data. I checked for missing values and duplicate category-year combinations.

The original dataset was already complete, so I did not remove any records or fill in any missing values. Each row in the cleaned dataset represents one spending category in one year.

In [ ]:
spending_tidy = spending_wide.melt(
    id_vars="Category",
    var_name="Year",
    value_name="Spending_Billions"
)

spending_tidy["Category"] = (
    spending_tidy["Category"].str.strip()
)

spending_tidy["Year"] = pd.to_numeric(
    spending_tidy["Year"],
    errors="raise"
).astype(int)

spending_tidy["Spending_Billions"] = pd.to_numeric(
    spending_tidy["Spending_Billions"],
    errors="raise"
)

spending_tidy = spending_tidy.sort_values(
    ["Category", "Year"]
).reset_index(drop=True)

assert len(spending_tidy) == 25
assert not spending_tidy.isna().any().any()
assert not spending_tidy.duplicated(
    ["Category", "Year"]
).any()

# Reconstruct the wide data to check that no values changed.
spending_roundtrip = spending_tidy.pivot(
    index="Category",
    columns="Year",
    values="Spending_Billions"
)

spending_original = spending_wide.set_index(
    "Category"
).copy()

spending_original.columns = (
    spending_original.columns.astype(int)
)

pd.testing.assert_frame_equal(
    spending_roundtrip.sort_index(),
    spending_original.sort_index(),
    check_names=False
)

print(spending_tidy.to_string(index=False))
print("\nData types:")
print(spending_tidy.dtypes.to_string())

           Category  Year  Spending_Billions
   Food and Tobacco  1940             22.200
   Food and Tobacco  1945             44.500
   Food and Tobacco  1950             59.600
   Food and Tobacco  1955             73.200
   Food and Tobacco  1960             86.800
Household Operation  1940             10.500
Household Operation  1945             15.500
Household Operation  1950             29.000
Household Operation  1955             36.500
Household Operation  1960             46.200
 Medical and Health  1940              3.530
 Medical and Health  1945              5.760
 Medical and Health  1950              9.710
 Medical and Health  1955             14.000
 Medical and Health  1960             21.100
      Personal Care  1940              1.040
      Personal Care  1945              1.980
      Personal Care  1950              2.450
      Personal Care  1955              3.400
      Personal Care  1960              5.400
  Private Education  1940              0.341
  Private 

In [ ]:
spending_comparison = spending_tidy.pivot(
    index="Category",
    columns="Year",
    values="Spending_Billions"
)[[1940, 1960]]

spending_comparison.columns = [
    "Spending_1940",
    "Spending_1960"
]

spending_comparison["Increase_Billions"] = (
    spending_comparison["Spending_1960"]
    - spending_comparison["Spending_1940"]
)

spending_comparison["Percentage_Increase"] = (
    100
    * spending_comparison["Increase_Billions"]
    / spending_comparison["Spending_1940"]
)

spending_comparison = spending_comparison.sort_values(
    "Percentage_Increase",
    ascending=False
)

print(spending_comparison.round(2).to_string())

print(
    "\nLargest percentage increase:",
    spending_comparison["Percentage_Increase"].idxmax()
)

print(
    "Highest spending in 1960:",
    spending_comparison["Spending_1960"].idxmax()
)

                     Spending_1940  Spending_1960  Increase_Billions  Percentage_Increase
Category                                                                                 
Private Education             0.34           3.64               3.30               967.45
Medical and Health            3.53          21.10              17.57               497.73
Personal Care                 1.04           5.40               4.36               419.23
Household Operation          10.50          46.20              35.70               340.00
Food and Tobacco             22.20          86.80              64.60               290.99

Largest percentage increase: Private Education
Highest spending in 1960: Food and Tobacco


### Conclusion

From what I found, private education had the largest percentage increase. Spending rose from $0.341 billion in 1940 to $3.64 billion in 1960, which was an increase of about 967.45%.

However, food and tobacco had the highest spending in 1960 at $86.80 billion. This shows that the category with the fastest percentage growth was different from the category with the highest spending amount.

The source documentation does not state an inflation adjustment, so I cannot describe these increases as inflation-adjusted growth. These five categories also do not represent all U.S. personal spending.

## Dataset 2: World Telephone Counts, 1951–1961

This dataset includes telephone counts for seven world regions in 1951 and 1956–1961. The values are measured in thousands of telephones.

Question: Which region had the largest absolute increase from 1951 to 1961, and which region had the largest percentage increase?

Online CSV:
https://vincentarelbundock.github.io/Rdatasets/csv/datasets/WorldPhones.csv

Official R documentation:
https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/WorldPhones.html

The documentation identifies McNeil’s Interactive Data Analysis: A Practical Primer, published in 1977, as the source.

In [ ]:
phones_wide = pd.read_csv(
    data_dir / "world_telephones_wide.csv"
)

check_wide(
    phones_wide,
    "Original wide telephone dataset"
)

Original wide telephone dataset
Shape: (7, 8)
Missing values: 0
Duplicate rows: 0
 Year  N.Amer  Europe  Asia  S.Amer  Oceania  Africa  Mid.Amer
 1951   45939   21574  2876    1815     1646      89       555
 1956   60423   29990  4708    2568     2366    1411       733
 1957   64721   32510  5230    2695     2526    1546       773
 1958   68484   35218  6662    2845     2691    1663       836
 1959   71799   37598  6856    3000     2868    1769       911
 1960   76036   40341  8220    3145     3054    1905      1008
 1961   79831   43173  9053    3338     3224    2005      1076


### Cleanup and Transformation

The original dataset stores each region in a separate column, and several region names are abbreviated. I used melt() to create Region and Telephones_Thousands columns. I then replaced the abbreviated region labels with full names.

The documentation describes Mid.Amer as Central America. I converted the years and telephone counts to integers, sorted the data, and checked for missing values and duplicate year-region combinations.

The dataset does not include 1952–1955. I kept the years that were provided without filling those gaps with zeros. Each cleaned row represents one region in one observed year.

In [ ]:
phones_tidy = phones_wide.melt(
    id_vars="Year",
    var_name="Region",
    value_name="Telephones_Thousands"
)

region_names = {
    "N.Amer": "North America",
    "Europe": "Europe",
    "Asia": "Asia",
    "S.Amer": "South America",
    "Oceania": "Oceania",
    "Africa": "Africa",
    "Mid.Amer": "Central America"
}

assert set(phones_tidy["Region"]) == set(region_names)

phones_tidy["Region"] = (
    phones_tidy["Region"].map(region_names)
)

phones_tidy["Year"] = pd.to_numeric(
    phones_tidy["Year"],
    errors="raise"
).astype(int)

phones_tidy["Telephones_Thousands"] = pd.to_numeric(
    phones_tidy["Telephones_Thousands"],
    errors="raise"
).astype(int)

phones_tidy = phones_tidy.sort_values(
    ["Region", "Year"]
).reset_index(drop=True)

assert len(phones_tidy) == 49
assert not phones_tidy.isna().any().any()
assert not phones_tidy.duplicated(
    ["Year", "Region"]
).any()

# Check that reshaping preserved every measurement.
phones_roundtrip = phones_tidy.pivot(
    index="Year",
    columns="Region",
    values="Telephones_Thousands"
)

phones_original = phones_wide.set_index(
    "Year"
).rename(columns=region_names)

pd.testing.assert_frame_equal(
    phones_roundtrip.sort_index(axis=1),
    phones_original.sort_index(axis=1),
    check_names=False
)

print(phones_tidy.to_string(index=False))

 Year          Region  Telephones_Thousands
 1951          Africa                    89
 1956          Africa                  1411
 1957          Africa                  1546
 1958          Africa                  1663
 1959          Africa                  1769
 1960          Africa                  1905
 1961          Africa                  2005
 1951            Asia                  2876
 1956            Asia                  4708
 1957            Asia                  5230
 1958            Asia                  6662
 1959            Asia                  6856
 1960            Asia                  8220
 1961            Asia                  9053
 1951 Central America                   555
 1956 Central America                   733
 1957 Central America                   773
 1958 Central America                   836
 1959 Central America                   911
 1960 Central America                  1008
 1961 Central America                  1076
 1951          Europe           

In [ ]:
phone_comparison = phones_tidy.pivot(
    index="Region",
    columns="Year",
    values="Telephones_Thousands"
)[[1951, 1961]]

phone_comparison.columns = [
    "Thousands_1951",
    "Thousands_1961"
]

phone_comparison["Increase_Thousands"] = (
    phone_comparison["Thousands_1961"]
    - phone_comparison["Thousands_1951"]
)

phone_comparison["Percentage_Increase"] = (
    100
    * phone_comparison["Increase_Thousands"]
    / phone_comparison["Thousands_1951"]
)

phone_comparison = phone_comparison.sort_values(
    "Increase_Thousands",
    ascending=False
)

print(phone_comparison.round(2).to_string())

print(
    "\nLargest absolute increase:",
    phone_comparison["Increase_Thousands"].idxmax()
)

print(
    "Largest percentage increase:",
    phone_comparison["Percentage_Increase"].idxmax()
)

                 Thousands_1951  Thousands_1961  Increase_Thousands  Percentage_Increase
Region                                                                                  
North America             45939           79831               33892                73.78
Europe                    21574           43173               21599               100.12
Asia                       2876            9053                6177               214.78
Africa                       89            2005                1916              2152.81
Oceania                    1646            3224                1578                95.87
South America              1815            3338                1523                83.91
Central America             555            1076                 521                93.87

Largest absolute increase: North America
Largest percentage increase: Africa


### Conclusion

North America had the largest absolute increase, adding 33,892 thousand telephones between 1951 and 1961. That equals 33,892,000 telephones.

Africa had the largest percentage increase at about 2,152.81%. Its telephone count increased from 89 thousand to 2,005 thousand. Its smaller starting value helps explain why its percentage increase was much larger.

All in all, absolute growth and percentage growth show different parts of the results. These numbers are telephone counts rather than rates per person, so this dataset alone cannot show which region had the greatest access relative to its population.

## Dataset 3: Virginia Death Rates in 1940

This dataset includes five age groups and four population groups: rural males, rural females, urban males, and urban females. The values are annual death rates per 1,000 people.

Question: Within each age group, how do rural and urban death rates compare for males and females?

Online CSV:
https://vincentarelbundock.github.io/Rdatasets/csv/datasets/VADeaths.csv

Official R documentation:
https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/VADeaths.html

The original source identified by R is Molyneaux, Gilliam, and Florant’s 1947 article, “Differences in Virginia Death Rates by Color, Sex, Age and Rural or Urban Residence,” in American Sociological Review.

DOI:
https://doi.org/10.2307/2086734

In [ ]:
deaths_wide = pd.read_csv(
    data_dir / "virginia_death_rates_wide.csv"
)

check_wide(
    deaths_wide,
    "Original wide Virginia death-rate dataset"
)

Original wide Virginia death-rate dataset
Shape: (5, 5)
Missing values: 0
Duplicate rows: 0
Age_Group  Rural Male  Rural Female  Urban Male  Urban Female
    50-54        11.7           8.7        15.4           8.4
    55-59        18.1          11.7        24.3          13.6
    60-64        26.9          20.3        37.0          19.3
    65-69        41.0          30.9        54.6          35.1
    70-74        66.0          54.3        71.1          50.0


### Cleanup and Transformation

The original column names combine two variables: residence and sex. I used melt() to place those columns into one Population_Group column. I then split that column into separate Residence and Sex columns.

I converted the death rates to numeric values and made the age groups an ordered category. After splitting Population_Group, I removed that combined column because its information was preserved in the two separate columns.

Each cleaned row represents one age group, one residence type, and one sex. I checked that all 20 original rates were preserved without missing values or duplicate group combinations.

In [ ]:
deaths_tidy = deaths_wide.melt(
    id_vars="Age_Group",
    var_name="Population_Group",
    value_name="Death_Rate_Per_1000"
)

deaths_tidy[["Residence", "Sex"]] = (
    deaths_tidy["Population_Group"].str.split(
        " ",
        n=1,
        expand=True
    )
)

deaths_tidy = deaths_tidy.drop(
    columns="Population_Group"
)

age_order = [
    "50-54",
    "55-59",
    "60-64",
    "65-69",
    "70-74"
]

deaths_tidy["Age_Group"] = pd.Categorical(
    deaths_tidy["Age_Group"],
    categories=age_order,
    ordered=True
)

deaths_tidy["Death_Rate_Per_1000"] = pd.to_numeric(
    deaths_tidy["Death_Rate_Per_1000"],
    errors="raise"
)

deaths_tidy = deaths_tidy[
    [
        "Age_Group",
        "Residence",
        "Sex",
        "Death_Rate_Per_1000"
    ]
]

deaths_tidy = deaths_tidy.sort_values(
    ["Age_Group", "Residence", "Sex"]
).reset_index(drop=True)

assert len(deaths_tidy) == 20
assert not deaths_tidy.isna().any().any()
assert not deaths_tidy.duplicated(
    ["Age_Group", "Residence", "Sex"]
).any()

# Rebuild the original groups to check all rates.
deaths_roundtrip = deaths_tidy.assign(
    Population_Group=(
        deaths_tidy["Residence"]
        + " "
        + deaths_tidy["Sex"]
    )
).pivot(
    index="Age_Group",
    columns="Population_Group",
    values="Death_Rate_Per_1000"
)

deaths_roundtrip.index = (
    deaths_roundtrip.index.astype(str)
)

pd.testing.assert_frame_equal(
    deaths_roundtrip.sort_index(axis=1),
    deaths_wide.set_index("Age_Group").sort_index(axis=1),
    check_names=False
)

print(deaths_tidy.to_string(index=False))

Age_Group Residence    Sex  Death_Rate_Per_1000
    50-54     Rural Female                  8.7
    50-54     Rural   Male                 11.7
    50-54     Urban Female                  8.4
    50-54     Urban   Male                 15.4
    55-59     Rural Female                 11.7
    55-59     Rural   Male                 18.1
    55-59     Urban Female                 13.6
    55-59     Urban   Male                 24.3
    60-64     Rural Female                 20.3
    60-64     Rural   Male                 26.9
    60-64     Urban Female                 19.3
    60-64     Urban   Male                 37.0
    65-69     Rural Female                 30.9
    65-69     Rural   Male                 41.0
    65-69     Urban Female                 35.1
    65-69     Urban   Male                 54.6
    70-74     Rural Female                 54.3
    70-74     Rural   Male                 66.0
    70-74     Urban Female                 50.0
    70-74     Urban   Male              

In [ ]:
death_comparison = deaths_tidy.pivot(
    index=["Age_Group", "Sex"],
    columns="Residence",
    values="Death_Rate_Per_1000"
)

death_comparison["Urban_Minus_Rural"] = (
    death_comparison["Urban"]
    - death_comparison["Rural"]
)

print("Rates and differences, per 1,000 people:\n")
print(death_comparison.round(2).to_string())

print("\nHighest rate in the dataset:")

highest_rate_row = deaths_tidy.loc[
    [deaths_tidy["Death_Rate_Per_1000"].idxmax()]
]

print(highest_rate_row.to_string(index=False))

male_differences = death_comparison.xs(
    "Male",
    level="Sex"
)["Urban_Minus_Rural"]

female_differences = death_comparison.xs(
    "Female",
    level="Sex"
)["Urban_Minus_Rural"]

print(
    "\nUrban male rate exceeds rural male rate "
    "in all five groups:",
    bool((male_differences > 0).all())
)

print(
    "Urban female rate exceeds rural female rate in",
    int((female_differences > 0).sum()),
    "of five groups"
)

Rates and differences, per 1,000 people:

Residence         Rural  Urban  Urban_Minus_Rural
Age_Group Sex                                    
50-54     Female    8.7    8.4               -0.3
          Male     11.7   15.4                3.7
55-59     Female   11.7   13.6                1.9
          Male     18.1   24.3                6.2
60-64     Female   20.3   19.3               -1.0
          Male     26.9   37.0               10.1
65-69     Female   30.9   35.1                4.2
          Male     41.0   54.6               13.6
70-74     Female   54.3   50.0               -4.3
          Male     66.0   71.1                5.1

Highest rate in the dataset:
Age_Group Residence  Sex  Death_Rate_Per_1000
    70-74     Urban Male                 71.1

Urban male rate exceeds rural male rate in all five groups: True
Urban female rate exceeds rural female rate in 2 of five groups


### Conclusion

From what I found, urban males had higher death rates than rural males in all five age groups. The pattern for females was mixed. Urban females had higher rates in the 55–59 and 65–69 groups, but lower rates in the 50–54, 60–64, and 70–74 groups.

The highest rate in the dataset was 71.1 deaths per 1,000 people for urban males ages 70–74. Death rates increased with age within each of the four population groups.

However, the data do not explain what caused these differences. Population denominators were not provided, so I did not average the rates and claim an overall population-weighted death rate.

## Exporting the Cleaned Datasets

The following code saves each cleaned dataset as a separate CSV file in a cleaned_data folder. The original wide CSV files are kept unchanged.

In [ ]:
cleaned_dir = data_dir / "cleaned_data"
cleaned_dir.mkdir(exist_ok=True)

cleaned_datasets = [
    ("us_spending_tidy.csv", spending_tidy),
    ("world_telephones_tidy.csv", phones_tidy),
    ("virginia_death_rates_tidy.csv", deaths_tidy)
]

for filename, frame in cleaned_datasets:
    frame.to_csv(
        cleaned_dir / filename,
        index=False
    )

    print(
        "Saved:",
        filename,
        "| rows:",
        len(frame)
    )

Saved: us_spending_tidy.csv | rows: 25
Saved: world_telephones_tidy.csv | rows: 49
Saved: virginia_death_rates_tidy.csv | rows: 20


## Overall Conclusion

This project showed me how wide datasets can make analysis harder when years, regions, or multiple variables are stored in column names. Using pandas helped me put those variables into separate columns while keeping the original information.

I also learned that interpreting the results correctly matters. Percentage changes differ from absolute changes, counts differ from rates, and historical patterns do not establish causes.

All in all, reshaping the datasets made it easier to compare groups and answer the analysis questions.